# CRAG식 교정 검색

검색 점수가 기준보다 낮을 때만 질문을 보정해 다시 검색합니다.

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** 이 절의 CRAG식 흐름은 첫 검색의 최고 점수를 임계값과 비교하고, 낮을 때만 질문을 한 번 보정해 재검색합니다. 평가 값, 분기 기준, 보정 질문, 두 번째 검색 결과를 모두 상태로 남겨 어느 단계가 결과를 바꿨는지 확인합니다.

**다른 상황에 적용.** 상품 검색에서 최고 유사도가 낮으면 모델명과 제조사를 보완해 한 번 더 검색하고, 충분하면 첫 후보를 그대로 사용할 수 있습니다. 재검색 횟수를 제한해야 끝없이 보정하는 흐름을 피할 수 있습니다.

**비교와 경계.** 원 논문의 CRAG에는 검색 결과 평가와 웹 검색 등 더 넓은 절차가 포함됩니다. 이 노트북은 임계값 기반 단일 보정만 구현한 축소 예제입니다. 유사도 0.55는 보편 기준이 아니며 임베딩 모델과 데이터에 따라 다시 정해야 합니다.


## 실행 환경 준비

고정한 Python 패키지를 확인합니다.

**이 셀의 코드 읽기.** `%pip install`은 이 노트북 커널이 사용할 패키지를 설치합니다. 앞 셀의 값을 입력으로 사용하지 않으며, 설치가 끝난 뒤의 코드 셀에서 패키지를 가져와 사용합니다.


In [1]:
%pip install -q langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3 numpy==2.5.3
import sys
print("준비 완료:", sys.version.split()[0])

Note: you may need to restart the kernel to use updated packages.
준비 완료: 3.12.13


## .env 찾기

가까운 .env에서 키를 읽고 존재 여부만 표시합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [2]:
# 현재 폴더에서 가까운 .env를 찾되 비밀 값은 출력하지 않습니다.
import os, time
from dotenv import find_dotenv, load_dotenv
env_path=find_dotenv(usecwd=True)
load_dotenv(env_path)
print("OPENAI_API_KEY:","있음" if os.getenv("OPENAI_API_KEY") else "없음")

OPENAI_API_KEY: 있음


## 공통 데이터

Ch12의 축소 데이터와 Ch11 방식의 정답 문서 ID를 사용합니다.

**이 셀의 코드 읽기.** `json.loads(text)`는 JSON 문자열을 파이썬 딕셔너리나 목록으로 변환해 반환합니다. `read_text()`는 파일 내용을 문자열로 반환합니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [3]:
# 배포 묶음에서도 같은 상대 경로를 찾도록 프로젝트 루트를 거슬러 올라갑니다.
import json
from pathlib import Path
import numpy as np
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"labs").is_dir())
data_dir=root/"labs/part4/ch14-rag-patterns/data"
docs=json.loads((data_dir/"documents.json").read_text(encoding="utf-8"))
queries=json.loads((data_dir/"queries.json").read_text(encoding="utf-8"))
print("문서·질문:",len(docs),len(queries))

문서·질문: 12 8


## 모델과 임베딩

모델과 256차원 문서 벡터를 준비합니다.

**이 셀의 코드 읽기.** `OpenAIEmbeddings(...)`는 문자열을 수치 벡터로 변환할 임베딩 클라이언트를 준비합니다. 실제 호출은 `embed_query()`나 `embed_documents()`에서 일어납니다. `embed_documents(texts)`는 여러 문장을 입력받아 문장 수만큼의 벡터를 반환합니다. `init_chat_model(...)`은 채팅 모델 객체를 만들며, 이 줄에서는 아직 답변을 생성하지 않습니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [4]:
# 256차원은 Ch12와 같은 비교 조건이며 행 하나가 문서 하나입니다.
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
chat=init_chat_model("gpt-4.1-mini",model_provider="openai",temperature=0)
embedder=OpenAIEmbeddings(model="text-embedding-3-small",dimensions=256)
vectors=np.array(embedder.embed_documents([d["text"] for d in docs]))
print("임베딩 배열:",vectors.shape)

임베딩 배열: (12, 256)


## 검색·생성 함수

검색 점수, 토큰, 시간, 추정 비용을 같은 방식으로 기록합니다.

**이 셀의 코드 읽기.** `search(question: str, k: int = 3)`, `ask(prompt: str)`, `show_hits(hits: list[tuple[dict[str, Any], float]])`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `invoke(input)`는 정의한 함수를 뒤에서 호출할 때 해당 함수 본문 안에서 모델이나 그래프를 실행하고 응답 객체를 반환합니다. `embed_query(text)`는 검색 질문 한 개를 벡터 하나로 반환합니다.


In [5]:
# 코사인 유사도는 내적을 두 벡터의 길이로 나누어 계산합니다.
from typing import Any
usage_log: list[dict[str, float | int]] = []

def search(question: str, k: int = 3) -> list[tuple[dict[str, Any], float]]:
    q=np.array(embedder.embed_query(question))
    scores=vectors@q/(np.linalg.norm(vectors,axis=1)*np.linalg.norm(q))
    # argsort 결과를 뒤집어 점수가 큰 문서부터 k개를 고릅니다.
    return [(docs[i],float(scores[i])) for i in np.argsort(scores)[::-1][:k]]
def ask(prompt: str) -> tuple[str, int, float]:
    started=time.perf_counter()
    msg=chat.invoke(prompt)
    usage=msg.usage_metadata or {}
    details=usage.get("input_token_details", {})
    input_tokens=usage.get("input_tokens", 0)
    cached_tokens=details.get("cache_read", 0)
    output_tokens=usage.get("output_tokens", 0)
    tokens=usage.get("total_tokens", 0)
    # 2026-09-10 gpt-4.1-mini 공개 단가로 추정하며 실제 청구액과 다를 수 있습니다.
    usd=((input_tokens-cached_tokens)*0.40+cached_tokens*0.10+output_tokens*1.60)/1_000_000
    usage_log.append({"input":input_tokens,"cached":cached_tokens,"output":output_tokens,"usd":usd})
    return msg.text,tokens,round((time.perf_counter()-started)*1000,1)

def show_hits(hits: list[tuple[dict[str, Any], float]]) -> list[tuple[str, float]]:
    return [(d["doc_id"],round(score,3)) for d,score in hits]

## 첫 검색과 품질 점수

원 질문의 첫 검색 결과에서 최고 유사도를 읽습니다. 0.55 임계값과 비교할 0.322를 확인합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [6]:
# 첫 검색의 최고 유사도를 고정 임계값과 비교할 값으로 저장합니다.
question="노쇼하면 방이 언제 풀리나요?"
first=search(question,k=2)
quality=first[0][1]
threshold=0.55
print("첫 검색:",show_hits(first),"최고 점수:",round(quality,3))

첫 검색: [('FAC-001', 0.322), ('LIB-001', 0.317)] 최고 점수: 0.322


## 임계값에 따른 보정

최고 점수가 임계값 미만이면 질문을 한 번 보정해 재검색합니다. 보정 질문과 바뀐 1위 문서를 확인합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [7]:
# 기준 미만일 때만 질문을 풀어 쓰고 재검색을 한 번 실행합니다.
if quality<threshold:
    corrected,tokens,latency=ask(f"도서관 규정 검색에 알맞게 용어를 풀어 질문 하나로 바꾸세요:{question}")
    final=search(corrected,k=3)
    action="재검색"
else: corrected,tokens,latency,final,action=question,0,0.0,first,"유지"
print(action,corrected)
print("최종:",show_hits(final),"토큰:",tokens,"ms:",latency)

재검색 노쇼 발생 시 예약한 도서관 방은 언제 자동으로 해제되나요?
최종: [('LIB-001', 0.451), ('LIB-004', 0.392), ('ACC-002', 0.335)] 토큰: 58 ms: 676.9


## 반복 상한

실제로 실행한 재검색 횟수를 상한과 비교합니다. 한 번을 넘으면 검사가 실패합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [8]:
# 재검색 횟수가 코드에 정한 한 번을 넘지 않았는지 검사합니다.
max_retries=1
retry_count=1 if action=="재검색" else 0
print("재검색 횟수:",retry_count,"/ 상한:",max_retries)
assert retry_count<=max_retries

재검색 횟수: 1 / 상한: 1
